# Notebook 06 — Campaign ROI Measurement
**Goal:** Quantify the financial return of the campaign.
Calculate CPA, channel-level ROI, incremental lift revenue, and run sensitivity analysis.
Translate analytics into a business-ready ROI framework.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')
from roi_calculator import (
    compute_roi, print_roi_report, channel_roi_comparison,
    segment_roi, incremental_lift_roi, sensitivity_analysis,
    plot_channel_roi, plot_sensitivity
)
from eda_utils import save

# Cost assumptions (adjust to reflect real business)
COST_PER_CONTACT = 50        # ₹50 per call (agent time + infra)
REVENUE_PER_CONVERSION = 5000  # ₹5,000 avg product revenue

df = pd.read_csv('../data/processed/segmented_data.csv')
print(f'Records: {len(df):,}')
print(f'Cost assumptions: ₹{COST_PER_CONTACT}/contact | ₹{REVENUE_PER_CONVERSION}/conversion')

## 1. Overall Campaign ROI

In [ ]:
overall = compute_roi(
    n_contacts=len(df),
    n_converted=df['subscribed'].sum(),
    cost_per_contact=COST_PER_CONTACT,
    revenue_per_conversion=REVENUE_PER_CONVERSION
)
print_roi_report(overall, 'Overall Campaign ROI')

## 2. Channel-Level ROI — Cellular vs Telephone

In [ ]:
channel_df = channel_roi_comparison(df, COST_PER_CONTACT, REVENUE_PER_CONVERSION)
print(channel_df[['channel','conv_rate_pct','roi_pct','cpa','total_revenue','profit']].to_string(index=False))

fig = plot_channel_roi(channel_df)
save(fig, '21_channel_roi.png')
plt.show()

## 3. Incremental Lift ROI — What Did Cellular Add?

In [ ]:
ctrl = df[df['contact']=='telephone']
trt = df[df['contact']=='cellular']

lift = incremental_lift_roi(
    control_n=len(ctrl), control_conv=ctrl['subscribed'].sum(),
    treatment_n=len(trt), treatment_conv=trt['subscribed'].sum(),
    cost_per_contact=COST_PER_CONTACT,
    revenue_per_conversion=REVENUE_PER_CONVERSION
)
print(f"\nIncremental Lift Analysis")
print(f"  Baseline (Telephone) rate  : {lift['baseline_rate']}%")
print(f"  Treatment (Cellular) rate  : {lift['treatment_rate']}%")
print(f"  Lift (percentage points)   : +{lift['lift_pp']}pp")
print(f"  Incremental conversions    : {lift['incremental_conversions']:,}")
print(f"  Incremental revenue        : ₹{lift['incremental_revenue']:,.0f}")
print(f"  Total campaign cost        : ₹{lift['total_cost']:,.0f}")
print(f"  Incremental ROI            : {lift['incremental_roi']:.1f}%")

## 4. Segment-Level ROI

In [ ]:
seg_df = segment_roi(df, COST_PER_CONTACT, REVENUE_PER_CONVERSION)
print('Segment ROI Breakdown:')
print(seg_df[['segment','conv_rate_pct','roi_pct','cpa','profit']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10,5))
colors = ['#1F4E79' if r > 0 else '#E76F51' for r in seg_df['roi_pct']]
bars = ax.bar(seg_df['segment'].astype(str), seg_df['roi_pct'], color=colors, width=0.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
import matplotlib.ticker as mtick
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('ROI by Customer Segment', fontsize=13, fontweight='bold')
ax.set_ylabel('ROI (%)')
ax.set_xlabel('Segment')
for bar, val in zip(bars, seg_df['roi_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'{val:.0f}%', ha='center', fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
save(fig, '22_segment_roi.png')
plt.show()

## 5. Sensitivity Analysis — What If Conversion Rate Changes?

In [ ]:
base_rate = df['subscribed'].mean()
sens = sensitivity_analysis(
    base_conv_rate=base_rate,
    n_contacts=10000,
    cost_per_contact=COST_PER_CONTACT,
    revenue_per_conversion=REVENUE_PER_CONVERSION
)

print(f'Base conversion rate: {base_rate*100:.2f}%')
print(sens[['conv_change_pct','conv_rate_pct','roi_pct','cpa']]
      .iloc[::4].to_string(index=False))

fig = plot_sensitivity(sens)
save(fig, '23_sensitivity_analysis.png')
plt.show()

## 6. Break-Even Analysis

In [ ]:
breakeven_rate = COST_PER_CONTACT / REVENUE_PER_CONVERSION * 100
print(f'Break-even conversion rate: {breakeven_rate:.2f}%')
print(f'Current conversion rate  : {base_rate*100:.2f}%')
margin = base_rate*100 - breakeven_rate
print(f'Safety margin            : +{margin:.2f}pp above break-even')

## 7. Executive Summary — Business Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════╗
║         CAMPAIGN ROI EXECUTIVE SUMMARY               ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  KEY FINDINGS                                        ║
║  1. Cellular channel delivers significantly higher   ║
║     ROI than telephone (statistically significant    ║
║     at 95% confidence level)                         ║
║                                                      ║
║  2. High-Value Responder segment contributes         ║
║     disproportionate ROI — prioritize for next       ║
║     campaign cycle                                   ║
║                                                      ║
║  3. Contact frequency >3 shows diminishing returns   ║
║     — cap contacts to 2-3 per customer to optimize  ║
║     CPA                                              ║
║                                                      ║
║  RECOMMENDATIONS                                     ║
║  → Shift 70%+ budget to cellular channel             ║
║  → Use XGBoost model scores to rank contact list     ║
║  → Contacting top 30% by score captures ~65%        ║
║    of conversions at ~50% lower cost                 ║
║  → De-prioritize Hard-to-Convert segment             ║
╚══════════════════════════════════════════════════════╝
""")